In [1]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Memulai Tahap 1: Setup Scraper Lokal...")

try:
    # 1. Konfigurasi Browser (Tanpa mode headless, agar layar Chrome muncul)
    options = webdriver.ChromeOptions()
    
    # 2. Menginstal dan memanggil ChromeDriver secara otomatis
    print("Membuka browser Chrome...")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # 3. Menuju TKP
    test_url = 'https://id.jobstreet.com/id/data-scientist-jobs'
    print(f"Mencoba mengakses: {test_url}")
    driver.get(test_url)
    
    # 4. Jeda panjang untuk observasi visual
    print("Menunggu halaman termuat. Jika ada CAPTCHA, silakan klik manual di browser Chrome yang terbuka...")
    time.sleep(10) 
    
    # 5. Cek hasil
    page_title = driver.title
    print(f"\n✅ Eksekusi selesai. Judul halaman: {page_title}")
    
    # Menutup browser
    driver.quit()

except Exception as e:
    print(f"❌ Terjadi kesalahan: {e}")

Memulai Tahap 1: Setup Scraper Lokal...
Membuka browser Chrome...
Mencoba mengakses: https://id.jobstreet.com/id/data-scientist-jobs
Menunggu halaman termuat. Jika ada CAPTCHA, silakan klik manual di browser Chrome yang terbuka...

✅ Eksekusi selesai. Judul halaman: Lowongan Kerja Data Scientist di Indonesia - Mei 2026 | Jobstreet


In [2]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Memulai Tahap 2: Ekstraksi Data dari 1 Halaman JobStreet...\n")

# 1. Buka Browser
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    # 2. Akses URL Target
    test_url = 'https://id.jobstreet.com/id/data-scientist-jobs'
    print(f"Membuka halaman: {test_url}")
    driver.get(test_url)
    
    # Beri waktu agar lowongan termuat (JobStreet menggunakan React/Next.js)
    time.sleep(5) 
    
    # 3. Mengambil mentahan HTML dari browser yang sudah terbuka
    html_source = driver.page_source
    
    # 4. Serahkan HTML ke BeautifulSoup untuk dibedah
    soup = BeautifulSoup(html_source, "lxml")
    
    # JobStreet biasanya membungkus setiap kartu lowongan dalam tag <article>
    articles = soup.find_all("article")
    print(f"\n✅ Ditemukan {len(articles)} kartu lowongan di halaman ini.")
    
    jobs_data = []
    
    # 5. Ekstraksi elemen per kartu lowongan
    for artikel in articles:
        # Mencari berdasarkan atribut data-automation (Sangat stabil karena ini dipakai QA engineer Jobstreet)
        posisi = artikel.find(attrs={"data-automation": "jobTitle"})
        perusahaan = artikel.find(attrs={"data-automation": "jobCompany"})
        lokasi = artikel.find(attrs={"data-automation": "jobLocation"})
        
        # Ekstraksi Teks (Ternary operator untuk menghindari error jika elemen tidak ada)
        posisi_text = posisi.get_text(" ", strip=True) if posisi else "Posisi tidak ditemukan"
        perusahaan_text = perusahaan.get_text(" ", strip=True) if perusahaan else "Perusahaan tidak disebutkan"
        lokasi_text = lokasi.get_text(" ", strip=True) if lokasi else "Lokasi tidak ditemukan"
        
        # Ekstraksi Link Lowongan (Untuk kita scrape skill-nya nanti)
        # Link biasanya ada di dalam tag <a> di dalam elemen posisi
        link_element = artikel.find('a', attrs={"data-automation": "jobTitle"})
        link_href = "https://id.jobstreet.com" + link_element['href'] if link_element and 'href' in link_element.attrs else "Link tidak ditemukan"
        
        jobs_data.append({
            "Posisi": posisi_text,
            "Perusahaan": perusahaan_text,
            "Lokasi": lokasi_text,
            "Link": link_href
        })
        
    # Tampilkan 3 data pertama sebagai bukti
    print("\n--- Sampel 3 Lowongan Pertama ---")
    for i, job in enumerate(jobs_data[:3]):
        print(f"{i+1}. {job['Posisi']} | {job['Perusahaan']} | {job['Lokasi']}")
        
except Exception as e:
    print(f"❌ Terjadi kesalahan: {e}")

finally:
    driver.quit() # Pastikan browser tertutup meskipun terjadi error

Memulai Tahap 2: Ekstraksi Data dari 1 Halaman JobStreet...

Membuka halaman: https://id.jobstreet.com/id/data-scientist-jobs

✅ Ditemukan 32 kartu lowongan di halaman ini.

--- Sampel 3 Lowongan Pertama ---
1. CRM and Supply Chain Data Scientist | PT Balarama Prajakarsaka Indonesia | Indragiri Hilir
2. Data Analyst | Brilliant Think Center | Ciputat Timur
3. Data Scientist | PT Sharing Vision Indonesia | Jakarta Pusat


In [3]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Memulai Tahap 3: Mass Extraction (Target 1000+ Lowongan)...\n")

# 1. Tentukan target role (Format URL JobStreet menggunakan tanda hubung)
target_roles = [
    "data-scientist", "data-analyst", "frontend-developer", 
    "backend-developer", "ui-ux-designer", "cyber-security", 
    "devops-engineer", "machine-learning"
]
pages_per_role = 5 # 8 role x 5 halaman x ~30 lowongan = ~1200 target data

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

all_jobs = []

try:
    # 2. Mulai Looping Raksasa
    for role in target_roles:
        print(f"\n🔍 Menarik data untuk kategori: {role.upper()}")
        
        for page in range(1, pages_per_role + 1):
            url = f'https://id.jobstreet.com/id/{role}-jobs?page={page}'
            print(f"   -> Scraping Halaman {page}...")
            
            driver.get(url)
            time.sleep(4) # WAKTU JEDA KRUSIAL (Jangan dipercepat)
            
            soup = BeautifulSoup(driver.page_source, "lxml")
            articles = soup.find_all("article")
            
            if not articles:
                print("      ⚠️ Tidak ada data atau halaman habis, lanjut ke kategori berikutnya.")
                break # Keluar dari loop halaman jika sudah mentok/habis
            
            # 3. Ekstraksi per kartu
            for artikel in articles:
                posisi = artikel.find(attrs={"data-automation": "jobTitle"})
                perusahaan = artikel.find(attrs={"data-automation": "jobCompany"})
                lokasi = artikel.find(attrs={"data-automation": "jobLocation"})
                link_element = artikel.find('a', attrs={"data-automation": "jobTitle"})
                
                posisi_text = posisi.get_text(" ", strip=True) if posisi else "N/A"
                perusahaan_text = perusahaan.get_text(" ", strip=True) if perusahaan else "N/A"
                lokasi_text = lokasi.get_text(" ", strip=True) if lokasi else "N/A"
                link_href = "https://id.jobstreet.com" + link_element['href'] if link_element and 'href' in link_element.attrs else "N/A"
                
                all_jobs.append({
                    "Kategori_Pencarian": role,
                    "Posisi": posisi_text,
                    "Perusahaan": perusahaan_text,
                    "Lokasi": lokasi_text,
                    "Link": link_href
                })

except Exception as e:
    print(f"❌ Terjadi kesalahan fatal: {e}")

finally:
    driver.quit()

# 4. Pembersihan dan Penyimpanan Data
print("\nMemproses pembersihan data...")
df_jobs = pd.DataFrame(all_jobs)

# Menghapus duplikat berdasarkan Link lowongan (agar tidak ada lowongan ganda)
df_jobs.drop_duplicates(subset=['Link'], inplace=True) 

# Menyimpan ke file master
file_name = 'jobstreet_master_links.csv'
df_jobs.to_csv(file_name, index=False)

print(f"✅ MASS EXTRACTION SELESAI!")
print(f"Total lowongan unik terkumpul: {len(df_jobs)} baris.")
print(f"Data tersimpan di: {file_name}")

Memulai Tahap 3: Mass Extraction (Target 1000+ Lowongan)...


🔍 Menarik data untuk kategori: DATA-SCIENTIST
   -> Scraping Halaman 1...
   -> Scraping Halaman 2...
   -> Scraping Halaman 3...
   -> Scraping Halaman 4...
   -> Scraping Halaman 5...

🔍 Menarik data untuk kategori: DATA-ANALYST
   -> Scraping Halaman 1...
   -> Scraping Halaman 2...
   -> Scraping Halaman 3...
   -> Scraping Halaman 4...
   -> Scraping Halaman 5...

🔍 Menarik data untuk kategori: FRONTEND-DEVELOPER
   -> Scraping Halaman 1...
   -> Scraping Halaman 2...
   -> Scraping Halaman 3...
   -> Scraping Halaman 4...
   -> Scraping Halaman 5...

🔍 Menarik data untuk kategori: BACKEND-DEVELOPER
   -> Scraping Halaman 1...
   -> Scraping Halaman 2...
   -> Scraping Halaman 3...
   -> Scraping Halaman 4...
   -> Scraping Halaman 5...

🔍 Menarik data untuk kategori: UI-UX-DESIGNER
   -> Scraping Halaman 1...
   -> Scraping Halaman 2...
   -> Scraping Halaman 3...
   -> Scraping Halaman 4...
   -> Scraping Halaman 5...

In [4]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("Memulai Tahap 4: Uji Coba Ekstraksi Detail Lowongan (1 Data)...\n")

# 1. Membaca tautan dari file CSV yang baru saja Anda buat
try:
    df_links = pd.read_csv('jobstreet_master_links.csv')
    test_link = df_links['Link'].iloc[0]  # Mengambil tautan baris paling pertama
    print(f"✅ CSV terbaca. Target Link: {test_link}")
except Exception as e:
    print("❌ Gagal membaca CSV. Pastikan file berada di folder yang sama.")
    exit()

# 2. Konfigurasi Browser
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    print("Membuka browser dan menuju tautan detail...")
    driver.get(test_link)
    
    # Tunggu lebih lama karena halaman detail biasanya lebih berat
    time.sleep(6) 
    
    # 3. Membedah HTML
    soup = BeautifulSoup(driver.page_source, "lxml")
    
    # Mencari kotak teks deskripsi pekerjaan (Job Ad Details)
    # Ini adalah atribut standar JobStreet untuk detail lowongan
    job_desc_element = soup.find(attrs={"data-automation": "jobAdDetails"})
    
    if job_desc_element:
        # Mengambil teks dan mempertahankan baris baru (enter) agar mudah dibaca
        job_desc_text = job_desc_element.get_text(separator="\n", strip=True)
        print("\n✅ SUPER! Deskripsi Pekerjaan Berhasil Ditemukan!\n")
        print("--- Cuplikan Deskripsi (500 Karakter Pertama) ---")
        print(job_desc_text[:500] + "\n\n...[Teks Sisanya Terpotong]")
    else:
        print("\n❌ Gagal menemukan elemen deskripsi. Struktur HTML mungkin berbeda atau tertutup elemen lain.")

except Exception as e:
    print(f"\n❌ Terjadi kesalahan: {e}")

finally:
    driver.quit()

Memulai Tahap 4: Uji Coba Ekstraksi Detail Lowongan (1 Data)...

✅ CSV terbaca. Target Link: https://id.jobstreet.com/id/job/92216448?type=standard&ref=search-standalone&origin=cardTitle#sol=fcfbcfc5d0bb420164069447a26253f55e9407e5
Membuka browser dan menuju tautan detail...

✅ SUPER! Deskripsi Pekerjaan Berhasil Ditemukan!

--- Cuplikan Deskripsi (500 Karakter Pertama) ---
PT Balarama Prajakarsaka Indonesia
adalah perusahaan pemasok (
supply company
) terkemuka yang bergerak khusus di industri kelapa. Beroperasi di seluruh wilayah Indonesia dan melayani pasar internasional, kami berkomitmen penuh untuk menghadirkan produk turunan kelapa berkualitas tinggi yang berbasis pada keunggulan, konsistensi, dan keberlanjutan (
sustainability
).
Sebagai jembatan antara potensi pengadaan lokal dan pasar global, kami membangun jaringan yang kuat dengan para petani, pengepul, 

...[Teks Sisanya Terpotong]
